# Simple Transformer for Translation (German → English)

This notebook explains the main idea from the original lab using simpler words and shorter code.

## The goal

Input: `Ein Hund spielt im Schnee.`  
Goal: produce an English sentence such as `A dog is playing in the snow.`

A Transformer translation model has two main parts:

1. **Encoder** reads the German sentence and builds a useful representation.
2. **Decoder** uses that representation to produce English one token at a time.

> This is a learning version of the original notebook. The data-loader helper and a trained model file are external files in the original lab, so the notebook explains where they are needed rather than pretending it is fully standalone.

## 1. Imports and device

PyTorch builds and runs the neural network. We use a GPU if one is available; otherwise we use the CPU.

In [1]:
import math
import torch
import torch.nn as nn
from torch import Tensor
from torch.nn import Transformer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", DEVICE)

Using: cpu


## 2. Special tokens

Words are converted to integer IDs before a model can process them.

- `<unk>`: a word not in the vocabulary
- `<pad>`: fills shorter sentences so batches have equal lengths
- `<bos>`: beginning of a sentence
- `<eos>`: end of a sentence

In [2]:
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
SPECIAL_TOKENS = ["<unk>", "<pad>", "<bos>", "<eos>"]

## 3. Load the Multi30K data

The dataset has already been downloaded and extracted into `multi30k_data/`.

Required files:
- `train.de`: German training sentences (source)
- `train.en`: English training sentences (target)
- `val.de`: German validation sentences
- `val.en`: English validation sentences

The next cell reads these files, builds German and English vocabularies, and creates `train_dataloader` and `val_dataloader`. No external helper `.py` file is needed.

In [3]:
import os

print("Current folder:", os.getcwd())
print("Files in folder:", os.listdir())

Current folder: d:\AI Engineering\deeplearningpytrochprojects\Learn BERT in Code
Files in folder: ['ENCODER-DECODER FOR TRANSLATION.IPYNB', 'EncoderBertModel.ipynb', 'multi30k_data', 'Multi30K_de_en_dataloader.py', 'Simple_Transformer_Translation_Explained.ipynb', 'Simple_Transformer_Translation_Explained_fixed.ipynb', '__pycache__']


In [4]:
# Load Multi30K files, build vocabularies, and create DataLoaders.
# Run this cell after downloading and extracting the dataset.

import os
import re
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

DATA_DIR = "multi30k_data"

required_files = ["train.de", "train.en", "val.de", "val.en"]
missing_files = [
    name for name in required_files
    if not os.path.isfile(os.path.join(DATA_DIR, name))
]
if missing_files:
    raise FileNotFoundError(
        f"Missing files in '{DATA_DIR}': {missing_files}. "
        "Check that you extracted training.tar.gz and validation.tar.gz."
    )

def read_lines(filename):
    path = os.path.join(DATA_DIR, filename)
    with open(path, encoding="utf-8") as file:
        return [line.strip() for line in file if line.strip()]

train_de = read_lines("train.de")
train_en = read_lines("train.en")
val_de = read_lines("val.de")
val_en = read_lines("val.en")

if len(train_de) != len(train_en):
    raise ValueError("train.de and train.en have different numbers of sentences.")
if len(val_de) != len(val_en):
    raise ValueError("val.de and val.en have different numbers of sentences.")

def tokenize(sentence):
    # Simple tokenizer: words and punctuation become separate tokens.
    return re.findall(r"\w+|[^\w\s]", sentence.lower())

class Vocabulary:
    def __init__(self, sentences, min_frequency=2):
        counts = Counter()
        for sentence in sentences:
            counts.update(tokenize(sentence))

        self.stoi = {token: index for index, token in enumerate(SPECIAL_TOKENS)}
        for word, count in counts.items():
            if count >= min_frequency and word not in self.stoi:
                self.stoi[word] = len(self.stoi)

    def __len__(self):
        return len(self.stoi)

    def encode(self, sentence):
        token_ids = [
            self.stoi.get(token, UNK_IDX)
            for token in tokenize(sentence)
        ]
        return [BOS_IDX] + token_ids + [EOS_IDX]

# German is the source language; English is the target language.
vocab_transform = {
    "de": Vocabulary(train_de),
    "en": Vocabulary(train_en),
}

text_transform = {
    "de": vocab_transform["de"].encode,
    "en": vocab_transform["en"].encode,
}

class TranslationDataset(Dataset):
    def __init__(self, source_sentences, target_sentences):
        self.source_sentences = source_sentences
        self.target_sentences = target_sentences

    def __len__(self):
        return len(self.source_sentences)

    def __getitem__(self, index):
        source = torch.tensor(text_transform["de"](self.source_sentences[index]), dtype=torch.long)
        target = torch.tensor(text_transform["en"](self.target_sentences[index]), dtype=torch.long)
        return source, target

def collate_batch(batch):
    sources, targets = zip(*batch)
    # Shape: [sequence_length, batch_size], as expected by this model.
    sources = pad_sequence(sources, batch_first=False, padding_value=PAD_IDX)
    targets = pad_sequence(targets, batch_first=False, padding_value=PAD_IDX)
    return sources, targets

train_dataset = TranslationDataset(train_de, train_en)
val_dataset = TranslationDataset(val_de, val_en)

train_dataloader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, collate_fn=collate_batch
)
val_dataloader = DataLoader(
    val_dataset, batch_size=32, shuffle=False, collate_fn=collate_batch
)

print("Training pairs:", len(train_dataset))
print("Validation pairs:", len(val_dataset))
print("German vocabulary:", len(vocab_transform["de"]))
print("English vocabulary:", len(vocab_transform["en"]))

# Show one example batch to verify the data.
source_batch, target_batch = next(iter(train_dataloader))
print("Source batch shape:", source_batch.shape)
print("Target batch shape:", target_batch.shape)


Training pairs: 29000
Validation pairs: 1014
German vocabulary: 7882
English vocabulary: 5898
Source batch shape: torch.Size([34, 32])
Target batch shape: torch.Size([41, 32])


## 4. Positional encoding

A Transformer needs help knowing word order. Positional encoding adds information about each token's position: first word, second word, and so on.

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, embedding_size, dropout=0.1, max_length=5000):
        super().__init__()
        positions = torch.arange(max_length).unsqueeze(1)
        scales = torch.exp(
            torch.arange(0, embedding_size, 2) * (-math.log(10000.0) / embedding_size)
        )

        pe = torch.zeros(max_length, 1, embedding_size)
        pe[:, 0, 0::2] = torch.sin(positions * scales)
        pe[:, 0, 1::2] = torch.cos(positions * scales)

        self.register_buffer("pe", pe)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(x + self.pe[:x.size(0)])

## 5. A smaller Encoder–Decoder Transformer

To make the original model easier to learn, we use a smaller embedding size and fewer layers. A smaller model is easier to experiment with, but it may translate less accurately.

- **Embedding** changes each token ID into a vector of numbers.
- **Encoder** reads the complete source sentence.
- **Decoder** predicts the target sentence using the source representation and previously generated English tokens.
- **Output layer** gives a score for every English vocabulary token.

In [6]:
class TranslationTransformer(nn.Module):
    def __init__(self, source_vocab_size, target_vocab_size,
                 embedding_size=128, heads=4, layers=2, feedforward_size=256):
        super().__init__()

        self.source_embedding = nn.Embedding(source_vocab_size, embedding_size, padding_idx=PAD_IDX)
        self.target_embedding = nn.Embedding(target_vocab_size, embedding_size, padding_idx=PAD_IDX)
        self.position = PositionalEncoding(embedding_size)

        self.transformer = Transformer(
            d_model=embedding_size,
            nhead=heads,
            num_encoder_layers=layers,
            num_decoder_layers=layers,
            dim_feedforward=feedforward_size,
            batch_first=False
        )
        self.output = nn.Linear(embedding_size, target_vocab_size)

    def forward(self, source, target_input, source_mask, target_mask,
                source_padding_mask, target_padding_mask):
        source = self.position(self.source_embedding(source) * math.sqrt(self.source_embedding.embedding_dim))
        target = self.position(self.target_embedding(target_input) * math.sqrt(self.target_embedding.embedding_dim))

        result = self.transformer(
            source, target,
            src_mask=source_mask,
            tgt_mask=target_mask,
            src_key_padding_mask=source_padding_mask,
            tgt_key_padding_mask=target_padding_mask,
            memory_key_padding_mask=source_padding_mask
        )
        return self.output(result)

## 6. Masks: stop the decoder from looking into the future

While predicting English, the decoder must not see the answer tokens that come later. For example, when predicting `playing`, it can use earlier words such as `A dog is`, but not the future word `snow`.

Padding masks also tell the model to ignore `<pad>` tokens.

In [7]:
def make_target_mask(length, device):
    # True above the diagonal means those future positions are blocked.
    return torch.triu(torch.ones(length, length, device=device, dtype=torch.bool), diagonal=1)

def make_masks(source, target_input):
    source_length = source.size(0)
    target_length = target_input.size(0)

    source_mask = torch.zeros(source_length, source_length, device=source.device, dtype=torch.bool)
    target_mask = make_target_mask(target_length, source.device)

    source_padding_mask = (source == PAD_IDX).transpose(0, 1)
    target_padding_mask = (target_input == PAD_IDX).transpose(0, 1)
    return source_mask, target_mask, source_padding_mask, target_padding_mask

## 7. How training examples are shifted

Suppose the correct English sentence is:

`<bos> A dog runs <eos>`

We feed the decoder the sentence without its last token:

`<bos> A dog runs`

The answers it must learn to predict are the sentence without its first token:

`A dog runs <eos>`

This lets the model learn the next token at every position.

In [8]:
# Example only (token IDs here are illustrative, not actual vocabulary IDs)
target = torch.tensor([[BOS_IDX], [10], [11], [12], [EOS_IDX]])
target_input = target[:-1]   # decoder input
target_answer = target[1:]   # correct next tokens

print("Decoder input shape:", target_input.shape)
print("Expected answers shape:", target_answer.shape)

Decoder input shape: torch.Size([4, 1])
Expected answers shape: torch.Size([4, 1])


## 8. Create the model

The vocabulary sizes come from the vocabularies built in Section 3. German is the source vocabulary and English is the target vocabulary.

In [9]:
# Create the model, optimizer, and loss function.
source_vocab_size = len(vocab_transform["de"])
target_vocab_size = len(vocab_transform["en"])

model = TranslationTransformer(
    source_vocab_size=source_vocab_size,
    target_vocab_size=target_vocab_size
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
loss_function = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

print("Model is ready.")
print("Source vocabulary size:", source_vocab_size)
print("Target vocabulary size:", target_vocab_size)


c:\Users\Furqan Khan\AppData\Local\miniconda3\envs\agent_env2\lib\site-packages\torch\nn\modules\transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Model is ready.
Source vocabulary size: 7882
Target vocabulary size: 5898


## 9. One training step, in plain language

For each batch:

1. Move German and English token IDs to the device.
2. Give the decoder the English sentence shifted right.
3. Build masks.
4. Ask the model to predict the English vocabulary scores.
5. Compare predictions with the correct next tokens using cross-entropy loss.
6. Backpropagate the error and update the model weights.

The DataLoader returns tensors shaped `[sequence_length, batch_size]`, which matches this notebook's Transformer.

In [10]:
def train_one_epoch(model, dataloader, optimizer, loss_function):
    model.train()
    total_loss = 0

    for source, target in dataloader:
        source = source.to(DEVICE)
        target = target.to(DEVICE)

        target_input = target[:-1]
        target_answer = target[1:]

        source_mask, target_mask, source_pad, target_pad = make_masks(source, target_input)

        predictions = model(
            source, target_input,
            source_mask, target_mask,
            source_pad, target_pad
        )

        # [sequence, batch, vocabulary] -> [sequence * batch, vocabulary]
        predictions = predictions.reshape(-1, predictions.size(-1))
        target_answer = target_answer.reshape(-1)

        loss = loss_function(predictions, target_answer)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

## 10. Training loop

Start with one or two epochs to test that the code works. Translation models often need much more training to become good. Training from scratch can take a long time, especially on a CPU.

In [11]:
# Start with 1 epoch to verify that training works.
# Increase the number of epochs after the first run succeeds.

for epoch in range(1):
    loss = train_one_epoch(model, train_dataloader, optimizer, loss_function)
    print(f"Epoch {epoch + 1}: training loss = {loss:.3f}")

torch.save(model.state_dict(), "simple_translation_model.pt")
print("Model saved to simple_translation_model.pt")


Epoch 1: training loss = 4.518
Model saved to simple_translation_model.pt


In [13]:
# German -> English prediction (run after training the model)
# You can test a sentence from validation data or type your own German sentence.

id_to_word_en = {index: word for word, index in vocab_transform["en"].stoi.items()}

def translate_german_to_english(german_sentence, max_length=50):
    model.eval()

    # Convert the German sentence into token IDs: [sequence_length, 1]
    source = torch.tensor(
        text_transform["de"](german_sentence), dtype=torch.long
    ).unsqueeze(1).to(DEVICE)

    generated_ids = [BOS_IDX]

    with torch.no_grad():
        for _ in range(max_length):
            # Decoder uses all English tokens generated so far.
            target_input = torch.tensor(
                generated_ids, dtype=torch.long
            ).unsqueeze(1).to(DEVICE)

            source_mask = torch.zeros(
                source.size(0), source.size(0),
                dtype=torch.bool, device=DEVICE
            )
            target_mask = make_target_mask(target_input.size(0), DEVICE)
            source_padding_mask = (source == PAD_IDX).transpose(0, 1)
            target_padding_mask = (target_input == PAD_IDX).transpose(0, 1)

            output = model(
                source, target_input,
                source_mask, target_mask,
                source_padding_mask, target_padding_mask
            )

            next_token = output[-1, 0].argmax(dim=-1).item()
            if next_token == EOS_IDX:
                break
            generated_ids.append(next_token)

    words = [
        id_to_word_en.get(token_id, "<unk>")
        for token_id in generated_ids[1:]
    ]
    return " ".join(words)


# Test with a German sentence from the validation dataset.
test_german = val_de[0]
print("German input:", test_german)
print("Predicted English:", translate_german_to_english(test_german))
print("Reference English:", val_en[0])

# Try your own sentence by changing this line:
print(translate_german_to_english("Ein Mann fährt Fahrrad."))


German input: Eine Gruppe von Männern lädt Baumwolle auf einen Lastwagen
Predicted English: a group of people are sitting on a street .
Reference English: A group of men are loading cotton onto a truck
a man is playing a red .


## 11. How translation happens (inference)

At translation time, the model does not receive the correct English sentence. It must build the answer itself:

1. Convert the German sentence to token IDs.
2. The encoder reads the German tokens once.
3. Start the decoder with `<bos>`.
4. Predict one English token.
5. Add that token to the decoder input.
6. Predict the next token.
7. Stop when the model predicts `<eos>` or reaches a maximum length.

This is called **autoregressive generation**: each new token depends on the tokens generated earlier.

## 12. Remember the whole idea

`German words → token IDs → embeddings + positions → encoder → decoder predicts English tokens one by one → English sentence`

The key concepts from the original notebook are embeddings, positional encoding, encoder/decoder, masks, shifted targets, cross-entropy loss, training, and greedy decoding.